# Comparativa_v0 — AG-CNN con MobileNetV2

Versión resumida para validar la propuesta principal del proyecto:

**Default vs Algoritmo Genético vs Random Search** usando una CNN con **Transfer Learning MobileNetV2**.

La idea es comprobar si el Algoritmo Genético puede encontrar una mejor configuración de hiperparámetros para clasificar enfermedades en hojas de papa.

## 1. Importación de librerías

Se cargan las librerías necesarias para manejar imágenes, entrenar modelos, ejecutar el Algoritmo Genético y visualizar resultados.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import keras
import torch

import ag_core as ag
from ag_core import HyperParams, Individual

print(f"GPU disponible: {'Si' if torch.cuda.is_available() else 'No'}")

## 2. Configuración general

Aquí se definen los parámetros principales del experimento: semilla, cantidad máxima de imágenes por clase, ruta del dataset, partición de datos y configuración del AG.

In [ ]:
SEED = 29
random.seed(SEED)
np.random.seed(SEED)

N_PER_CLASS = 200
DIR_DATASET = Path("data/PlantVillage")
SPLIT = (0.70, 0.15, 0.15)

BACKBONE = "MobileNetV2"

CLASS_NAMES = sorted([d.name for d in DIR_DATASET.iterdir() if d.is_dir()])
N_CLASS = len(CLASS_NAMES)

ag.configure(
    models={
        "MobileNetV2": {
            "fn": keras.applications.MobileNetV2,
            "preprocess": keras.applications.mobilenet_v2.preprocess_input,
        },
    },
    n_class=N_CLASS,
    seed=SEED,
    pop_size=8,
    gen_max=5,
    gen_pat=3,
    epochs_fitness=4,
    epochs_testeo=12,
)

print("Clases detectadas:", CLASS_NAMES)
print("Numero de clases:", N_CLASS)

## 3. Carga y división del dataset

Se cargan las imágenes de PlantVillage, se redimensionan a 224x224 y se dividen en entrenamiento, validación y prueba.

In [ ]:
def load_images(data_dir):
    X, y = [], []

    for idx, cname in enumerate(CLASS_NAMES):
        folder = data_dir / cname
        files = []
        for ext in ["*.JPG", "*.jpg", "*.JPEG", "*.jpeg", "*.PNG", "*.png"]:
            files.extend(folder.glob(ext))

        files = sorted(files)

        if len(files) > N_PER_CLASS:
            files = random.sample(files, N_PER_CLASS)

        print(f"{cname:30s} -> usando {len(files)} imagenes")

        for p in files:
            img = Image.open(p).convert("RGB").resize((224, 224))
            X.append(np.asarray(img, dtype=np.uint8))
            y.append(idx)

    return np.array(X), np.array(y)


X_raw, y_all = load_images(DIR_DATASET)

train_frac, valid_frac, test_frac = SPLIT

X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw,
    y_all,
    test_size=(valid_frac + test_frac),
    stratify=y_all,
    random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=test_frac / (valid_frac + test_frac),
    stratify=y_temp,
    random_state=SEED
)

cw = compute_class_weight("balanced", classes=np.arange(N_CLASS), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}

fit_data = dict(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    class_weight=CLASS_WEIGHT
)

print("\nResumen de datos:")
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print("Class weight:", CLASS_WEIGHT)

## 4. Configuraciones a comparar

Se comparan tres enfoques:

1. **Default:** configuración manual común.
2. **AG:** búsqueda evolutiva de hiperparámetros.
3. **Random Search:** búsqueda aleatoria con número similar de evaluaciones.

In [ ]:
DEFAULT_CHR = HyperParams(0.001, 32, "adam", 0.2)

print("Configuracion default:")
print(DEFAULT_CHR)

## 5. Ejecución de la comparativa

En esta sección se entrena y evalúa cada estrategia.  
El AG busca automáticamente una combinación de hiperparámetros que mejore el rendimiento en validación.

In [ ]:
all_results = {}

print("=" * 70)
print(f"BACKBONE: {BACKBONE}")
print("=" * 70)

ag.clear_fitness_cache()

# 1) Evaluación default
print("\n[1/3] Evaluando configuracion DEFAULT...")
default_indiv = Individual(DEFAULT_CHR)

t0 = time.time()
default_indiv.set_fitness(
    ag.get_fitness(BACKBONE, DEFAULT_CHR, **fit_data)
)
default_time = time.time() - t0

print(f"Fitness default: {default_indiv.fitness:.4f}")
print(f"Tiempo default: {default_time:.1f}s")

# 2) Algoritmo Genético
print("\n[2/3] Ejecutando ALGORITMO GENETICO...")
np.random.seed(SEED)
random.seed(SEED)

population = ag.init_population(ag.POP_SIZE)
ag_best, ag_curve, ag_stats, ag_time = ag.genetic_algo(
    BACKBONE,
    population,
    **fit_data
)

print("\nMejor cromosoma AG:")
print(ag_best.chromosome)
print(f"Fitness AG: {ag_best.fitness:.4f}")
print(f"Tiempo busqueda AG: {ag_time:.1f}s")

# 3) Random Search
print("\n[3/3] Ejecutando RANDOM SEARCH...")
n_evals = ag.POP_SIZE * len(ag_curve)

rs_best, rs_curve, rs_time = ag.random_search(
    BACKBONE,
    n_evals,
    **fit_data
)

print("\nMejor cromosoma Random Search:")
print(rs_best.chromosome)
print(f"Fitness Random Search: {rs_best.fitness:.4f}")
print(f"Tiempo busqueda RS: {rs_time:.1f}s")

search_time = {
    "Default": default_time,
    "Algoritmo Genetico": ag_time,
    "Random Search": rs_time,
}

all_results[BACKBONE] = {
    "default": default_indiv,
    "ag_best": ag_best,
    "ag_curve": ag_curve,
    "ag_stats": ag_stats,
    "rs_best": rs_best,
    "rs_curve": rs_curve,
    "search_time": search_time,
}

## 6. Evaluación final en test

Después de la búsqueda, se entrena/evalúa cada configuración en el conjunto de prueba.  
El test se usa solo al final para medir el rendimiento real del modelo.

In [ ]:
strategies = {
    "Default": DEFAULT_CHR,
    "Algoritmo Genetico": all_results[BACKBONE]["ag_best"].chromosome,
    "Random Search": all_results[BACKBONE]["rs_best"].chromosome,
}

test_results = {}
preds = {}

for name, chrom in strategies.items():
    print("\n" + "-" * 60)
    print(f"Evaluando estrategia: {name}")
    print("Cromosoma:", chrom)

    f1, acc, y_pred, train_time = ag.evaluate_test(
        BACKBONE,
        chrom,
        X_train,
        y_train,
        X_test,
        y_test,
        CLASS_WEIGHT
    )

    test_results[name] = {
        "f1": f1,
        "acc": acc,
        "train_time_test": train_time,
    }
    preds[name] = y_pred

    print(f"{name:20s} | F1-macro={f1:.4f} | Accuracy={acc:.4f} | Tiempo final={train_time:.1f}s")

all_results[BACKBONE]["test_results"] = test_results
all_results[BACKBONE]["preds"] = preds

## 7. Tabla resumen de resultados

Esta tabla permite comparar el rendimiento de la configuración manual, Random Search y el Algoritmo Genético.

In [ ]:
rows = []

for name in ["Default", "Random Search", "Algoritmo Genetico"]:
    if name == "Default":
        chrom = DEFAULT_CHR
        fitness_val = all_results[BACKBONE]["default"].fitness
    elif name == "Random Search":
        chrom = all_results[BACKBONE]["rs_best"].chromosome
        fitness_val = all_results[BACKBONE]["rs_best"].fitness
    else:
        chrom = all_results[BACKBONE]["ag_best"].chromosome
        fitness_val = all_results[BACKBONE]["ag_best"].fitness

    rows.append({
        "Estrategia": name,
        "Learning rate": chrom.alpha,
        "Batch size": chrom.batch,
        "Optimizador": chrom.phi,
        "Dropout": chrom.rho,
        "F1 val": fitness_val,
        "F1 test": test_results[name]["f1"],
        "Accuracy test": test_results[name]["acc"],
        "Tiempo busqueda (s)": search_time[name],
        "Tiempo entrenamiento final (s)": test_results[name]["train_time_test"],
    })

df_results = pd.DataFrame(rows)
df_results

## 8. Curva de convergencia

Se grafica cómo mejora el mejor fitness durante la búsqueda del AG y de Random Search.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    np.arange(1, len(ag_curve) + 1) * ag.POP_SIZE,
    ag_curve,
    marker="o",
    label="Algoritmo Genetico"
)

plt.plot(
    np.arange(1, len(rs_curve) + 1),
    rs_curve,
    marker="s",
    linestyle="--",
    label="Random Search"
)

plt.axhline(default_indiv.fitness, linestyle=":", label="Default")

plt.xlabel("Numero de evaluaciones")
plt.ylabel("Mejor fitness en validacion")
plt.title("Convergencia de la busqueda de hiperparametros")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. Matrices de confusión

Las matrices permiten observar en qué clases se equivoca más cada estrategia.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, ["Default", "Random Search", "Algoritmo Genetico"]):
    cm = confusion_matrix(y_test, preds[name])
    display_labels = [c.replace("Potato___", "") for c in CLASS_NAMES]

    ConfusionMatrixDisplay(
        cm,
        display_labels=display_labels
    ).plot(cmap="Blues", ax=ax, colorbar=False)

    ax.set_title(name)

plt.suptitle(f"Matrices de confusion — {BACKBONE}")
plt.tight_layout()
plt.show()

## 10. Interpretación rápida

Usa esta sección para comentar los resultados obtenidos.

Puntos a revisar:

- Si el AG obtuvo mayor F1 test que Default, entonces la búsqueda evolutiva mejoró la configuración manual.
- Si Random Search queda cerca del AG, se puede indicar que el espacio de búsqueda es pequeño o que el baseline ya era fuerte.
- Si el AG no gana, se puede justificar que se trabajó con pocas generaciones y poca población para reducir tiempo computacional.
- La matriz de confusión ayuda a identificar qué enfermedades se confunden más.

In [ ]:
print("Mejor configuracion encontrada por AG:")
print(all_results[BACKBONE]["ag_best"].chromosome)

print("\nTabla final:")
display(df_results)